# HAIO 2024 - Transzfer Tanulás Megoldás

Ez a notebook a Magyar Mesterséges Intelligencia Olimpia (HAIO) 2024-es transzfer tanulás feladatának megoldását tartalmazza.

A feladat lényege: CIFAR-100-on tanított ResNet modell tudásának átvitele CIFAR-10-re.

**Altaszkok:**
1. Transzfer tanítás előtt - modellek betanítása és kiértékelése
2. Transzfer tanítás - utolsó réteg finomhangolása
3. Modellek távolsága - L2 távolság mátrix
4. Több réteg továbbtanítása
5. Kevés adaton való tanítás

## Előkészítés

Importok, eszközbeállítás és segédfüggvények.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset

# GPU használata, ha elérhető
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Eszköz: {device}')

In [ ]:
# Adataugmentáció és normalizálás
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# CIFAR-10 adathalmazok
cifar10_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
cifar10_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
# CIFAR-10 train augmentáció nélkül (kiértékeléshez)
cifar10_train_eval = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_test)

cifar10_trainloader = DataLoader(cifar10_train, batch_size=128, shuffle=True, num_workers=2)
cifar10_testloader = DataLoader(cifar10_test, batch_size=128, shuffle=False, num_workers=2)
cifar10_train_evalloader = DataLoader(cifar10_train_eval, batch_size=128, shuffle=False, num_workers=2)

# CIFAR-100 adathalmazok
cifar100_train = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
cifar100_test = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

cifar100_trainloader = DataLoader(cifar100_train, batch_size=128, shuffle=True, num_workers=2)
cifar100_testloader = DataLoader(cifar100_test, batch_size=128, shuffle=False, num_workers=2)

print(f'CIFAR-10 tanító: {len(cifar10_train)}, teszt: {len(cifar10_test)}')
print(f'CIFAR-100 tanító: {len(cifar100_train)}, teszt: {len(cifar100_test)}')

## CIFAR-hez igazított ResNet modell

A standard ResNet-18 ImageNet-re lett tervezve (224x224 képek). A CIFAR adathalmaz 32x32 pixeles képekből áll,
ezért egy kisebb, CIFAR-ra optimalizált ResNet architektúrát használunk:
- Kezdeti konvolúció: 3x3, stride=1 (nincs maxpool)
- 3 csoport reziduális blokk: [16, 32, 64] csatorna
- Globális átlagos pooling az FC réteg előtt

In [ ]:
class BasicBlock(nn.Module):
    """Alap reziduális blokk 2 konvolúciós réteggel."""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut kapcsolat méreteltérés esetén
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class CIFARResNet(nn.Module):
    """CIFAR-ra optimalizált ResNet.
    
    3 csoport reziduális blokk [16, 32, 64] csatornával.
    Csoportonként `num_blocks` blokk (alapértelmezés: 3, ami ResNet-20-nak felel meg).
    """

    def __init__(self, num_blocks=3, num_classes=10):
        super().__init__()
        self.in_channels = 16

        # Kezdeti konvolúció - 3x3, stride=1, nincs maxpool
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # 3 reziduális csoport
        self.layer1 = self._make_layer(16, num_blocks, stride=1)   # 32x32
        self.layer2 = self._make_layer(32, num_blocks, stride=2)   # 16x16
        self.layer3 = self._make_layer(64, num_blocks, stride=2)   # 8x8

        # Globális átlagos pooling és osztályozó réteg
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

        # Súlyok inicializálása
        self._initialize_weights()

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


# Teszt: modell létrehozása és összefoglaló
test_model = CIFARResNet(num_blocks=3, num_classes=10)
total_params = sum(p.numel() for p in test_model.parameters())
print(f'CIFAR ResNet-20 paraméterek száma: {total_params:,}')
del test_model

In [ ]:
def evaluate(model, dataloader, device):
    """Modell kiértékelése: pontosság számítás."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total


def train_model(model, trainloader, testloader, epochs, lr=0.1, device='cuda',
                extra_testloader=None, extra_test_name=None):
    """Modell tanítása SGD optimalizálóval és cosine annealing ütemezővel.
    
    Visszaadja a tanítási és teszt pontosság/veszteség történetét.
    Ha extra_testloader meg van adva, azon is kiértékeli epochonként.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {
        'train_loss': [], 'train_acc': [],
        'test_acc': []
    }
    if extra_testloader is not None:
        history['extra_test_acc'] = []

    for epoch in range(epochs):
        # Tanítás
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        scheduler.step()

        train_loss = running_loss / total
        train_acc = 100.0 * correct / total
        test_acc = evaluate(model, testloader, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        log_str = f'Epoch {epoch+1}/{epochs} | Veszteség: {train_loss:.4f} | Tanító acc: {train_acc:.2f}% | Teszt acc: {test_acc:.2f}%'

        if extra_testloader is not None:
            extra_acc = evaluate(model, extra_testloader, device)
            history['extra_test_acc'].append(extra_acc)
            log_str += f' | {extra_test_name} acc: {extra_acc:.2f}%'

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(log_str)

    return history

---
## 1. feladat: Transzfer tanítás előtt

Ebben a részben:
1. Betanítunk egy ResNet modellt CIFAR-100-on
2. Betanítunk egy ResNet modellt CIFAR-10-en
3. Kicseréljük a CIFAR-100 modell utolsó FC rétegét a CIFAR-10 modell FC rétegére
4. Kiértékeljük a hibrid modellt CIFAR-10-en

In [ ]:
# CIFAR-100 modell betanítása
print('=== CIFAR-100 modell tanítása ===')
model_cifar100 = CIFARResNet(num_blocks=3, num_classes=100)
history_cifar100 = train_model(model_cifar100, cifar100_trainloader, cifar100_testloader,
                                epochs=50, lr=0.1, device=device)

cifar100_test_acc = evaluate(model_cifar100, cifar100_testloader, device)
print(f'\nCIFAR-100 modell végső teszt pontosság: {cifar100_test_acc:.2f}%')

In [ ]:
# CIFAR-10 modell betanítása
print('=== CIFAR-10 modell tanítása ===')
model_cifar10 = CIFARResNet(num_blocks=3, num_classes=10)
history_cifar10 = train_model(model_cifar10, cifar10_trainloader, cifar10_testloader,
                               epochs=50, lr=0.1, device=device)

cifar10_test_acc = evaluate(model_cifar10, cifar10_testloader, device)
print(f'\nCIFAR-10 modell végső teszt pontosság: {cifar10_test_acc:.2f}%')

In [ ]:
# Tanulási görbék ábrázolása
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Veszteség
axes[0].plot(history_cifar100['train_loss'], label='CIFAR-100 tanító')
axes[0].plot(history_cifar10['train_loss'], label='CIFAR-10 tanító')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Veszteség')
axes[0].set_title('Tanítási veszteség')
axes[0].legend()
axes[0].grid(True)

# Pontosság
axes[1].plot(history_cifar100['test_acc'], label='CIFAR-100 teszt')
axes[1].plot(history_cifar10['test_acc'], label='CIFAR-10 teszt')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pontosság (%)')
axes[1].set_title('Teszt pontosság')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Hibrid modell: CIFAR-100 törzs + CIFAR-10 FC réteg
print('=== Hibrid modell létrehozása ===')
model_hybrid = copy.deepcopy(model_cifar100)

# Az utolsó FC réteg cseréje a CIFAR-10 modell FC rétegére
model_hybrid.fc = copy.deepcopy(model_cifar10.fc)
model_hybrid = model_hybrid.to(device)

# Kiértékelés CIFAR-10-en
hybrid_train_acc = evaluate(model_hybrid, cifar10_train_evalloader, device)
hybrid_test_acc = evaluate(model_hybrid, cifar10_testloader, device)

print(f'Hibrid modell CIFAR-10 tanító pontosság: {hybrid_train_acc:.2f}%')
print(f'Hibrid modell CIFAR-10 teszt pontosság: {hybrid_test_acc:.2f}%')
print(f'\nÖsszehasonlítás:')
print(f'  Eredeti CIFAR-10 modell teszt acc: {cifar10_test_acc:.2f}%')
print(f'  Hibrid modell teszt acc:           {hybrid_test_acc:.2f}%')
print(f'\nA hibrid modell gyengén teljesít, mert a CIFAR-100-on tanult jellemzők')
print(f'és a CIFAR-10 FC réteg súlyai nincsenek összehangolva.')

---
## 2. feladat: Transzfer tanítás

A CIFAR-100-on előtanított modell utolsó rétegét lecseréljük egy új, véletlenszerűen inicializált
10 kimenetes FC rétegre. Az összes réteget befagyasztjuk az utolsó kivételével,
majd finomhangoljuk CIFAR-10-en.

Közben figyeljük a pontosságot mind CIFAR-10, mind CIFAR-100 teszthalmazon.

In [ ]:
# Transzfer tanulás: csak az utolsó réteget tanítjuk
print('=== Transzfer tanítás: utolsó réteg finomhangolása ===')

# CIFAR-100 modell másolása és utolsó réteg cseréje
model_transfer = copy.deepcopy(model_cifar100)
model_transfer.fc = nn.Linear(64, 10)  # Új véletlenszerű FC réteg 10 kimenettel

# Minden réteg befagyasztása az utolsó FC kivételével
for name, param in model_transfer.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

# Ellenőrzés: mennyi paramétert tanítunk
trainable = sum(p.numel() for p in model_transfer.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_transfer.parameters())
print(f'Tanítható paraméterek: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# Finomhangolás
history_transfer = train_model(
    model_transfer, cifar10_trainloader, cifar10_testloader,
    epochs=50, lr=0.1, device=device,
    extra_testloader=cifar100_testloader,
    extra_test_name='CIFAR-100'
)

transfer_test_acc = evaluate(model_transfer, cifar10_testloader, device)
print(f'\nTranszfer modell CIFAR-10 teszt pontosság: {transfer_test_acc:.2f}%')

In [ ]:
# Pontosság ábrázolása: CIFAR-10 és CIFAR-100 teszt
fig, ax = plt.subplots(figsize=(10, 6))

epochs_range = range(1, len(history_transfer['test_acc']) + 1)
ax.plot(epochs_range, history_transfer['test_acc'], 'b-o', markersize=3,
        label='CIFAR-10 teszt pontosság')
ax.plot(epochs_range, history_transfer['extra_test_acc'], 'r-s', markersize=3,
        label='CIFAR-100 teszt pontosság')

ax.set_xlabel('Epoch')
ax.set_ylabel('Pontosság (%)')
ax.set_title('Transzfer tanítás: pontosság alakulása\n(csak utolsó réteg tanítva)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f'Megfigyelés: A CIFAR-10 pontosság növekszik, míg a CIFAR-100 pontosság')
print(f'alacsony marad (hiszen az FC réteget CIFAR-10-re cseréltük).')
print(f'A befagyasztott jellemzőkinyerő rétegek CIFAR-100-on tanult reprezentációi')
print(f'jól használhatók CIFAR-10 osztályozásra is - ez a transzfer tanulás lényege.')

---
## 3. feladat: Modellek távolsága

Kiszámítjuk a páronkénti L2 (euklideszi) távolságot 3 modell paraméterei között:
1. Véletlenszerűen inicializált modell (10 kimenet)
2. CIFAR-100-on tanított modell az 1. feladatban kicserélt utolsó réteggel (hibrid)
3. A 2. feladatban finomhangolt transzfer modell

A távolságot az összes paraméter egybefűzött vektorán számítjuk.

In [ ]:
def get_param_vector(model):
    """Modell összes paraméterének egyetlen vektorba fűzése."""
    return torch.cat([p.data.view(-1) for p in model.parameters()])


def pairwise_l2_distance(models, names):
    """Páronkénti L2 távolság mátrix számítása és megjelenítése."""
    n = len(models)
    vectors = [get_param_vector(m).cpu() for m in models]
    dist_matrix = torch.zeros(n, n)

    for i in range(n):
        for j in range(n):
            dist_matrix[i, j] = torch.norm(vectors[i] - vectors[j], p=2).item()

    return dist_matrix


# Véletlenszerűen inicializált modell (10 kimenet, mint a transzfer modellé)
model_random = CIFARResNet(num_blocks=3, num_classes=10).to(device)

# A CIFAR-100 modell kicserélt utolsó réteggel (1. feladat hibrid modellje)
# A model_hybrid-ot használjuk (CIFAR-100 törzs + CIFAR-10 FC)
# De a feladatnak megfelelően inkább a CIFAR-100 modellt vesszük új random FC-vel
model_cifar100_swapped = copy.deepcopy(model_cifar100)
model_cifar100_swapped.fc = nn.Linear(64, 10)  # Új random 10-kimenetes FC
model_cifar100_swapped = model_cifar100_swapped.to(device)

# A 3 modell összehasonlítása
models = [model_random, model_cifar100_swapped, model_transfer]
names = ['Véletlenszerű', 'CIFAR-100 (cserélt FC)', 'Finomhangolt']

dist_matrix = pairwise_l2_distance(models, names)

# Megjelenítés
print('Páronkénti L2 távolság mátrix:')
print(f'{"":>25s}', end='')
for name in names:
    print(f'{name:>25s}', end='')
print()
for i, name in enumerate(names):
    print(f'{name:>25s}', end='')
    for j in range(len(names)):
        print(f'{dist_matrix[i, j]:>25.2f}', end='')
    print()

# Hőtérkép
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(dist_matrix.numpy(), cmap='YlOrRd')
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_yticklabels(names)

# Értékek megjelenítése a cellákon
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{dist_matrix[i, j]:.1f}',
                ha='center', va='center', fontsize=12)

ax.set_title('Modellek páronkénti L2 távolsága')
plt.colorbar(im, ax=ax, label='L2 távolság')
plt.tight_layout()
plt.show()

print(f'\nMegfigyelések:')
print(f'- A véletlenszerű modell messze van mindkét tanított modelltől.')
print(f'- A CIFAR-100 előtanított és a finomhangolt modell közel vannak egymáshoz,')
print(f'  mivel csak az utolsó réteget módosítottuk a finomhangolás során.')

---
## 4. feladat: Több réteg továbbtanítása

Most több réteget is feloldunk a finomhangoláshoz:
- **A) Második reziduális csoporttól:** layer2 + layer3 + fc
- **B) Második reziduális csoport utántól:** layer3 + fc

Összehasonlítjuk a teszt pontosságokat.

In [ ]:
def create_transfer_model(base_model, unfreeze_from='fc'):
    """Transzfer modell létrehozása megadott rétegektől feloldva.
    
    Args:
        base_model: CIFAR-100 előtanított modell
        unfreeze_from: 'fc', 'layer3', 'layer2', 'layer1', 'all'
    """
    model = copy.deepcopy(base_model)
    model.fc = nn.Linear(64, 10)  # Új FC réteg CIFAR-10-hez

    # Alapértelmezetten minden befagyasztva
    for param in model.parameters():
        param.requires_grad = False

    # Rétegek feloldása a megadott ponttól
    layers_order = ['layer1', 'layer2', 'layer3', 'fc']
    if unfreeze_from == 'all':
        for param in model.parameters():
            param.requires_grad = True
    else:
        start_idx = layers_order.index(unfreeze_from)
        for layer_name in layers_order[start_idx:]:
            layer = getattr(model, layer_name)
            for param in layer.parameters():
                param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Feloldva: {unfreeze_from}-tól | Tanítható: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)')

    return model

In [ ]:
# A) Második reziduális csoporttól feloldva (layer2 + layer3 + fc)
print('=== A) layer2-től feloldva ===')
model_unfreeze_layer2 = create_transfer_model(model_cifar100, unfreeze_from='layer2')
history_unfreeze_layer2 = train_model(
    model_unfreeze_layer2, cifar10_trainloader, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_layer2 = evaluate(model_unfreeze_layer2, cifar10_testloader, device)
print(f'Végső teszt pontosság (layer2-től): {acc_layer2:.2f}%')

In [ ]:
# B) Harmadik reziduális csoporttól feloldva (layer3 + fc)
print('=== B) layer3-tól feloldva ===')
model_unfreeze_layer3 = create_transfer_model(model_cifar100, unfreeze_from='layer3')
history_unfreeze_layer3 = train_model(
    model_unfreeze_layer3, cifar10_trainloader, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_layer3 = evaluate(model_unfreeze_layer3, cifar10_testloader, device)
print(f'Végső teszt pontosság (layer3-tól): {acc_layer3:.2f}%')

In [ ]:
# Összehasonlítás
fig, ax = plt.subplots(figsize=(10, 6))

epochs_range = range(1, 51)
ax.plot(epochs_range, history_transfer['test_acc'], label='Csak FC réteg', marker='.')
ax.plot(epochs_range, history_unfreeze_layer3['test_acc'], label='layer3 + FC', marker='.')
ax.plot(epochs_range, history_unfreeze_layer2['test_acc'], label='layer2 + layer3 + FC', marker='.')

ax.set_xlabel('Epoch')
ax.set_ylabel('CIFAR-10 teszt pontosság (%)')
ax.set_title('Több réteg finomhangolásának hatása')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f'\nÖsszefoglaló:')
print(f'  Csak FC réteg:         {transfer_test_acc:.2f}%')
print(f'  layer3 + FC:           {acc_layer3:.2f}%')
print(f'  layer2 + layer3 + FC:  {acc_layer2:.2f}%')
print(f'  Eredeti CIFAR-10:      {cifar10_test_acc:.2f}%')
print(f'\nMegfigyelés: Több réteg feloldásával általában jobb pontosság érhető el,')
print(f'de a túl sok feloldott réteg esetén elveszíthetjük az előtanítás előnyeit.')

---
## 5. feladat: Kevés adaton való tanítás

Megvizsgáljuk, hogyan teljesít a transzfer tanulás, ha csak az adatok egy részét használjuk:
- 50%-os CIFAR-10 tanító adathalmaz
- 20%-os CIFAR-10 tanító adathalmaz

Kérdés: El tudjuk-e érni a teljes adaton tanított CIFAR-10 modell pontosságát?

In [ ]:
def create_subset_loader(dataset, fraction, batch_size=128):
    """Adathalmaz egy részhalmazából DataLoader létrehozása."""
    n = len(dataset)
    indices = np.random.permutation(n)[:int(n * fraction)]
    subset = Subset(dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=2)


np.random.seed(42)  # Reprodukálhatóság

# 50%-os adathalmaz
cifar10_train_50pct = create_subset_loader(cifar10_train, 0.5)
print(f'50%-os tanító: {int(len(cifar10_train) * 0.5)} minta')

# 20%-os adathalmaz
cifar10_train_20pct = create_subset_loader(cifar10_train, 0.2)
print(f'20%-os tanító: {int(len(cifar10_train) * 0.2)} minta')

In [ ]:
# Transzfer tanulás 50% adattal (layer2-től feloldva, mert az volt a legjobb)
print('=== Transzfer tanulás 50% adattal ===')
model_50pct = create_transfer_model(model_cifar100, unfreeze_from='layer2')
history_50pct = train_model(
    model_50pct, cifar10_train_50pct, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_50pct = evaluate(model_50pct, cifar10_testloader, device)
print(f'Végső teszt pontosság (50% adat): {acc_50pct:.2f}%')

In [ ]:
# Transzfer tanulás 20% adattal
print('=== Transzfer tanulás 20% adattal ===')
model_20pct = create_transfer_model(model_cifar100, unfreeze_from='layer2')
history_20pct = train_model(
    model_20pct, cifar10_train_20pct, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_20pct = evaluate(model_20pct, cifar10_testloader, device)
print(f'Végső teszt pontosság (20% adat): {acc_20pct:.2f}%')

In [ ]:
# Összehasonlításként: nulláról tanítás korlátozott adattal
print('=== Nulláról tanítás 50% adattal (összehasonlítás) ===')
model_scratch_50pct = CIFARResNet(num_blocks=3, num_classes=10)
history_scratch_50pct = train_model(
    model_scratch_50pct, cifar10_train_50pct, cifar10_testloader,
    epochs=50, lr=0.1, device=device
)
acc_scratch_50pct = evaluate(model_scratch_50pct, cifar10_testloader, device)
print(f'Nulláról tanított (50% adat) teszt pontosság: {acc_scratch_50pct:.2f}%')

print('\n=== Nulláról tanítás 20% adattal (összehasonlítás) ===')
model_scratch_20pct = CIFARResNet(num_blocks=3, num_classes=10)
history_scratch_20pct = train_model(
    model_scratch_20pct, cifar10_train_20pct, cifar10_testloader,
    epochs=50, lr=0.1, device=device
)
acc_scratch_20pct = evaluate(model_scratch_20pct, cifar10_testloader, device)
print(f'Nulláról tanított (20% adat) teszt pontosság: {acc_scratch_20pct:.2f}%')

In [ ]:
# Összehasonlító ábra
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, 51)

# 50% adat
axes[0].plot(epochs_range, history_50pct['test_acc'], label='Transzfer (50%)')
axes[0].plot(epochs_range, history_scratch_50pct['test_acc'], label='Nulláról (50%)', linestyle='--')
axes[0].axhline(y=cifar10_test_acc, color='green', linestyle=':', label=f'Teljes CIFAR-10 ({cifar10_test_acc:.1f}%)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CIFAR-10 teszt pontosság (%)')
axes[0].set_title('50% tanító adattal')
axes[0].legend()
axes[0].grid(True)

# 20% adat
axes[1].plot(epochs_range, history_20pct['test_acc'], label='Transzfer (20%)')
axes[1].plot(epochs_range, history_scratch_20pct['test_acc'], label='Nulláról (20%)', linestyle='--')
axes[1].axhline(y=cifar10_test_acc, color='green', linestyle=':', label=f'Teljes CIFAR-10 ({cifar10_test_acc:.1f}%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('CIFAR-10 teszt pontosság (%)')
axes[1].set_title('20% tanító adattal')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Végső összefoglaló táblázat
print('=' * 70)
print('VÉGSŐ ÖSSZEFOGLALÓ')
print('=' * 70)
print(f'{"Módszer":<45s} {"Teszt acc (%)":>15s}')
print('-' * 70)
print(f'{"CIFAR-10 nulláról (100% adat)":<45s} {cifar10_test_acc:>14.2f}%')
print(f'{"CIFAR-100 nulláról":<45s} {cifar100_test_acc:>14.2f}%')
print(f'{"Hibrid (CIFAR-100 törzs + CIFAR-10 FC)":<45s} {hybrid_test_acc:>14.2f}%')
print(f'{"Transzfer: csak FC":<45s} {transfer_test_acc:>14.2f}%')
print(f'{"Transzfer: layer3 + FC":<45s} {acc_layer3:>14.2f}%')
print(f'{"Transzfer: layer2 + layer3 + FC":<45s} {acc_layer2:>14.2f}%')
print(f'{"Transzfer: 50% adat (layer2-től)":<45s} {acc_50pct:>14.2f}%')
print(f'{"Nulláról: 50% adat":<45s} {acc_scratch_50pct:>14.2f}%')
print(f'{"Transzfer: 20% adat (layer2-től)":<45s} {acc_20pct:>14.2f}%')
print(f'{"Nulláról: 20% adat":<45s} {acc_scratch_20pct:>14.2f}%')
print('=' * 70)

print(f'\nKövetkeztetések:')
print(f'1. A CIFAR-100-on tanult jellemzők jól transzferálhatók CIFAR-10-re.')
print(f'2. Több réteg feloldásával jobb eredmények érhetők el.')
print(f'3. Transzfer tanulással kevesebb adattal is versenyképes pontosság érhető el.')
print(f'4. Kevés adat esetén a transzfer tanulás előnye még kifejezettebb.')